# Train GPT-BERT on native (non-translated) Hindi/Telugu data

Trains a monolingual GPT-BERT from scratch on the CC-100-derived native data (`pulipakav-1/hi-te`), using this project's existing `babybabellm-gptbert` training code (`Babylm2026/gpt-bert/{hindi,telugu}`), scaled down to fit a single Colab GPU in a few hours instead of the original multi-GPU cluster-scale run (`max_steps=15625`, `global_batch_size=32768` across 3-4 GPUs).

**Set `LANG` below to `"hindi"` or `"telugu"` and run all cells top to bottom.**

Steps: clone repo -> download native data -> train a fresh tokenizer on it -> shard/tokenize the data -> train the model.

In [ ]:
LANG = "hindi"  # "hindi" or "telugu"
assert LANG in ("hindi", "telugu")

In [ ]:
# Cell 1: clone the repo and install dependencies
!git clone https://github.com/vishnup22/BabyLM.git
%cd BabyLM
!git checkout evaluation
!git pull
%cd Babylm2026/gpt-bert/{LANG}
!pip install -q -r requirements.txt

In [ ]:
# Cell 2: log in to Hugging Face (needed to pull pulipakav-1/hi-te)
from google.colab import userdata
import os

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")  # use whatever secret name you saved your token under
!hf auth whoami

In [ ]:
# Cell 3: download the native data and lay it out in the structure the tokenizer/shard
# tools expect: data/raw/<dataset_name>/<dataset_name>.train.<lang>.txt
from pathlib import Path
from huggingface_hub import hf_hub_download

LANG_CODE = {"hindi": "hi", "telugu": "te"}[LANG]
DATASET_NAME = f"native-{LANG}"

src_path = hf_hub_download(repo_id="pulipakav-1/hi-te", filename=f"{LANG}.txt", repo_type="dataset")

raw_dir = Path("data/raw") / DATASET_NAME
raw_dir.mkdir(parents=True, exist_ok=True)
dest_path = raw_dir / f"{DATASET_NAME}.train.{LANG_CODE}.txt"
dest_path.write_bytes(Path(src_path).read_bytes())
print(f"Placed {dest_path} ({dest_path.stat().st_size:,} bytes)")

In [ ]:
# Cell 4: train a fresh tokenizer on the native data (vocab_size=16384, matching the
# monolingual config -- reusing the old translated-data tokenizer here would reintroduce
# the tokenizer-granularity issue found earlier in this project)
!python tools/train_tokenizer_local.py \
  --dataset {DATASET_NAME} \
  --data_root data/raw \
  --output tokenizers/tokenizer_native_16384.json \
  --vocab_size 16384

In [ ]:
# Cell 5: tokenize the data and write train/valid shards (2% held out for validation)
!python tools/prepare_local_shards.py \
  --dataset {DATASET_NAME} \
  --data_root data/raw \
  --tokenizer tokenizers/tokenizer_native_16384.json \
  --output_base data/processed \
  --valid_fraction 0.02

In [ ]:
# Cell 6: train, single GPU. Hyperparameters below are SCALED DOWN from the original
# cluster-scale run (max_steps=15625, global_batch_size=32768 across 3-4 GPUs) to fit a
# single Colab GPU in a few hours -- adjust MAX_STEPS after watching the first ~50 steps'
# pace if you want to run longer or shorter. Model architecture (hidden_size, layers, etc.)
# is unchanged from configs/base.json -- only the training budget is reduced.
import os
os.environ["WANDB_MODE"] = "disabled"  # no W&B account needed

MAX_STEPS = 2000
GLOBAL_BATCH_SIZE = 256
LOCAL_BATCH_SIZE = 64  # lower this if you hit an out-of-memory error on a smaller GPU
SEQ_LENGTH = 128

%cd pretraining
!python train_single_gpu.py \
  --train_path ../data/processed/train \
  --valid_path ../data/processed/valid \
  --config_file ../configs/base.json \
  --tokenizer_path ../tokenizers/tokenizer_native_16384.json \
  --name native-{LANG}-gptbert \
  --output_dir ../model_checkpoints \
  --hybrid_numerator 2 \
  --hybrid_denominator 3 \
  --global_batch_size {GLOBAL_BATCH_SIZE} \
  --local_batch_size {LOCAL_BATCH_SIZE} \
  --seq_length {SEQ_LENGTH} \
  --max_steps {MAX_STEPS} \
  --save_every 200 \
  --validate_every 0 \
  --seed 42

## Optional: push the trained checkpoint to Hugging Face

Once training finishes, `../model_checkpoints/native-{LANG}-gptbert_2_3.bin` (plus `_ema.bin` and `_state_dict.bin`) will contain the trained weights. Run the cell below to upload them.

In [ ]:
# Cell 7 (optional): push the checkpoint + tokenizer + config to a new HF repo
from huggingface_hub import HfApi

REPO_ID = f"pulipakav-1/native-{LANG}-gptbert"
api = HfApi()
api.create_repo(repo_id=REPO_ID, repo_type="model", exist_ok=True)

for local_path, repo_path in [
    (f"../model_checkpoints/native-{LANG}-gptbert_2_3_ema.bin", "model_ema.bin"),
    ("../tokenizers/tokenizer_native_16384.json", "tokenizer.json"),
    ("../configs/base.json", "config_base.json"),
]:
    api.upload_file(path_or_fileobj=local_path, path_in_repo=repo_path, repo_id=REPO_ID, repo_type="model")

print(f"Pushed to https://huggingface.co/{REPO_ID}")